# NoiseChisel on a GPU with noisechisel-torch (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PabloMSanAla/noisechisel/blob/main/notebooks/colab_gpu.ipynb)

Runs [noisechisel-torch](https://github.com/PabloMSanAla/noisechisel) on one image with
the GPU and shows the detection mask, Sky and Sky STD with
[Pixel Clarity](https://pypi.org/project/pixel-clarity/).

**Runtime → Change runtime type → GPU** before running.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Install

* `noisechisel-torch[fits]` from PyPI (astropy for FITS files).
* One CuPy build matching the driver's CUDA version, for fast labelling on the GPU.
  Colab may already have a different one; mismatched builds are removed.
* `pixel-clarity`, the image viewer.

**If this cell removes a CuPy package, restart the session** (Runtime → Restart session)
before continuing. Restart also if the viewer in section 6 does not appear.

In [ ]:
import re, subprocess, sys, importlib.metadata as md

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
driver_cuda = int(re.search(r"CUDA Version:\s*(\d+)", smi).group(1))
want = f"cupy-cuda{min(driver_cuda, 13)}x"
have = []
for d in ("cupy", "cupy-cuda11x", "cupy-cuda12x", "cupy-cuda13x"):
    try:
        md.version(d); have.append(d)
    except md.PackageNotFoundError:
        pass
removed = [d for d in have if d != want]
if removed:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", *removed], check=True)
pkgs = ["noisechisel-torch[fits]", "pixel-clarity", "gdown"] + ([] if want in have else [want])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
print("driver CUDA", driver_cuda, "| CuPy:", want)
if removed:
    print(">>> Removed", removed, "- restart the session (Runtime > Restart session) now.")

## 3. Check noisechisel-torch and CuPy

In [ ]:
import noisechisel_torch as nct
from noisechisel_torch import NoiseChisel
nc = NoiseChisel(device="cuda")
print("noisechisel-torch", nct.__version__, "| device:", nc.device, "| labelling:", nc.ccl)
assert nc.ccl == "cupy", "CuPy not found: labelling would go through the CPU (slower)"

## 4. Get an image

A simulated Roman WFI frame (4088×4088) from Google Drive. To use your own image,
upload it (Files panel on the left) and set `PATH` to it.

In [ ]:
import os, gdown
DRIVE_ID = "1KVOIrK4gzOQQ_UwNTRicQDWzzD_Vvp-E"
PATH = "/content/image.fits"
if not os.path.exists(PATH):
    gdown.download(id=DRIVE_ID, output=PATH, quiet=False)

## 5. Run NoiseChisel on the GPU

In [ ]:
import time
from astropy.io import fits

img, header = fits.getdata(PATH, ext=0, header=True)
print("image", img.shape, img.dtype)

nc(img)                                   # first call: CUDA start-up and kernel compilation
torch.cuda.synchronize()
t0 = time.perf_counter()
res = nc(img)
torch.cuda.synchronize()
print(f"NoiseChisel on the GPU: {time.perf_counter() - t0:.2f} s")
res.info()

Options keep Gnuastro's names, e.g. `NoiseChisel(device="cuda", tilesize=(40, 40), qthresh=0.35)`.
The command line does the same and writes a FITS file like `astnoisechisel`:

In [ ]:
!noisechisel-torch {PATH} -h 0 -o /content/image_detected.fits --device cuda

## 6. Show the mask, Sky and Sky STD

The results go into an `HDUList` (one HDU per image). Pixel Clarity shows it as a stack:
move the slider to switch between the input, detections, Sky and Sky STD.

In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()     # Colab: allow third-party widgets
except ImportError:
    pass
import numpy as np
import pixel_clarity as pc

hdul = fits.HDUList([
    fits.PrimaryHDU(img, header=header),
    fits.ImageHDU(res.detections.cpu().numpy().astype(np.uint8), name="DETECTIONS"),
    fits.ImageHDU(res.sky.cpu().numpy(), name="SKY"),
    fits.ImageHDU(res.std.cpu().numpy(), name="SKY_STD"),
])
pc.show(hdul, kind="stack", names=["INPUT", "DETECTIONS", "SKY", "SKY_STD"])